In [1]:
# Imports
import sys
from pathlib import Path
import zipfile

import pandas as pd
import numpy as np

# Project Imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils

In [2]:
# =========================================================
# Input Paths
# =========================================================

PATH_BENCHMARK = (
    cfg.RAW_DIR / "benchmark/all_runs.csv"
)
df_bench = pd.read_csv(PATH_BENCHMARK)

PATH_BASELINE = (
    cfg.DATA_DIR / "metadata/baseline.csv"
)
df_baseline = pd.read_csv(PATH_BASELINE)

# =========================================================
# Output Paths
# =========================================================
OUTPUT_PATH = (
    cfg.PROCESSED_DIR / "all_runs_new"
)

In [3]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Select the five benchmark solvers 
# ============================================================

solvers = [
    "BURER2002",
    "DAv2",
    "DAv3",
    "MERZ1999GLS",
    "PALUBECKIS2004bMST2"
]

mqlib_solvers = [
    "BURER2002",
    "MERZ1999GLS",
    "PALUBECKIS2004bMST2"
]

df_clean = df_bench[
    df_bench["heuristic"].isin(solvers)
].copy()


# ============================================================
# 2. Merge instance-specific baseline time limits
# ============================================================

df_clean = df_clean.merge(
    df_baseline[["name", "limit"]],
    on="name",
    how="inner",
    validate="many_to_one"
)


# ============================================================
# 3. Extract configured time limit from filename
# ============================================================

df_clean["file_limit"] = pd.to_numeric(
    df_clean["file"].astype(str).str.extract(
        r"-([0-9]+(?:\.[0-9]+)?)-\d+\.csv$"
    )[0],
    errors="coerce"
)


# ============================================================
# 4. Calculate DAv3's adjusted time limit
#    Using the 3-second offset from the preceding benchmark
# ============================================================

baseline = df_clean["limit"]

df_clean["dav3_target_limit"] = np.maximum(
    1,
    np.minimum(
        np.floor(baseline),
        np.floor(1.1 * baseline - 3)
    )
)


# ============================================================
# 5. Select the correct experimental batches
# ============================================================

# MQLib: configured TL must match baseline TL
is_mqlib = df_clean["heuristic"].isin(mqlib_solvers)

mqlib_match = np.isclose(
    df_clean["file_limit"],
    df_clean["limit"],
    atol=0.0011,
    rtol=0
)

# DAv3: configured TL must match the 3-second-offset formula
is_dav3 = df_clean["heuristic"] == "DAv3"

dav3_match = np.isclose(
    df_clean["file_limit"],
    df_clean["dav3_target_limit"],
    atol=0.0011,
    rtol=0
)

# DAv2: retain its original runs
is_dav2 = df_clean["heuristic"] == "DAv2"

df_selected = df_clean[
    (is_mqlib & mqlib_match)
    | (is_dav3 & dav3_match)
    | is_dav2
].copy()


# ============================================================
# 6. Validate exactly five runs per solver-instance pair
# ============================================================

batch_check = (
    df_selected
    .groupby(["name", "heuristic"])
    .agg(
        n_runs=("seed", "size"),
        n_seeds=("seed", "nunique"),
        n_files=("file", "nunique")
    )
    .reset_index()
)

problems = batch_check[
    (batch_check["n_runs"] != 5)
    | (batch_check["n_seeds"] != 5)
    | (batch_check["n_files"] != 1)
]

if not problems.empty:
    display(problems)
    raise ValueError(
        "Some solver-instance pairs do not contain "
        "exactly five runs from one file."
    )

# Verify every selected solver-instance pair is valid
assert (batch_check["n_runs"] == 5).all()
assert (batch_check["n_seeds"] == 5).all()
assert (batch_check["n_files"] == 1).all()


# ============================================================
# 7. Check coverage and missing solver-instance pairs
# ============================================================

print("Selected dataset summary:")

display(
    df_selected.groupby("heuristic")
    .agg(
        instances=("name", "nunique"),
        runs=("name", "size"),
        files=("file", "nunique")
    )
    .reindex(solvers)
)

print("Total selected runs:", len(df_selected))
print("Unique instances:", df_selected["name"].nunique())
print("Missing filename limits:", df_selected["file_limit"].isna().sum())


# ============================================================
# 8. Create final dataset with original columns
# ============================================================

df_all_new = (
    df_selected[
        [
            "file",
            "name",
            "heuristic",
            "seed",
            "objective",
            "runtime",
            "solution"
        ]
    ]
    .sort_values(["heuristic", "name", "seed"])
    .reset_index(drop=True)
)


# ============================================================
# 9. Save the new CSV
# ============================================

df_all_new.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved dataset to:", OUTPUT_PATH)
print("Final shape:", df_all_new.shape)

Selected dataset summary:


,instances,runs,files
heuristic,,,
BURER2002,3396,16980,3396
DAv2,3319,16595,3319
DAv3,3400,17000,3400
MERZ1999GLS,3400,17000,3400
PALUBECKIS2004bMST2,3400,17000,3400


Total selected runs: 84575
Unique instances: 3400
Missing filename limits: 0

Saved dataset to: /Users/sargun/Desktop/DAMB-DAv3c/data/processed/all_runs_new
Final shape: (84575, 7)


In [4]:
# Compare instance coverage before and after filtering
coverage_before = (
    df_clean.groupby("heuristic")["name"]
    .apply(set)
)

coverage_after = (
    df_selected.groupby("heuristic")["name"]
    .apply(set)
)

for solver in solvers:
    before = coverage_before.get(solver, set())
    after = coverage_after.get(solver, set())

    missing = sorted(before - after)

    print(f"\n{solver}")
    print("Instances before:", len(before))
    print("Instances after :", len(after))
    print("Excluded instances:", len(missing))

    if missing:
        print("Excluded names:", missing)


BURER2002
Instances before: 3396
Instances after : 3396
Excluded instances: 0

DAv2
Instances before: 3319
Instances after : 3319
Excluded instances: 0

DAv3
Instances before: 3400
Instances after : 3400
Excluded instances: 0

MERZ1999GLS
Instances before: 3400
Instances after : 3400
Excluded instances: 0

PALUBECKIS2004bMST2
Instances before: 3400
Instances after : 3400
Excluded instances: 0


In [5]:
df_check = df_selected.copy()

df_check["runtime_ratio"] = (
    df_check["runtime"] / df_check["limit"]
)

summary = (
    df_check.groupby("heuristic")
    .agg(
        runs=("name", "size"),
        instances=("name", "nunique"),
        over_5pct=("runtime_ratio", lambda x: (x > 1.05).sum()),
        over_10pct=("runtime_ratio", lambda x: (x > 1.10).sum()),
        over_2x=("runtime_ratio", lambda x: (x > 2).sum()),
        median_ratio=("runtime_ratio", "median"),
        p95_ratio=("runtime_ratio", lambda x: x.quantile(0.95)),
        max_ratio=("runtime_ratio", "max")
    )
)

display(summary.round(4))

,runs,instances,over_5pct,over_10pct,over_2x,median_ratio,p95_ratio,max_ratio
heuristic,,,,,,,,
BURER2002,16980,3396,315,35,0,1.0056,1.0345,1.2109
DAv2,16595,3319,2823,2301,1227,0.9940,3.3039,23.5072
DAv3,17000,3400,11225,10809,9681,3.9284,411.4143,2395.2316
MERZ1999GLS,17000,3400,6,0,0,1.0005,1.0105,1.0690
PALUBECKIS2004bMST2,17000,3400,3042,1636,13,1.0126,1.2192,3.1214


In [6]:
reviewer_instances = [
    "g003215", "g000989", "g001269", "g002569",
    "g000377", "g001086", "g002440", "g001337",
    "g001883", "g000283", "imgseg_376020",
    "imgseg_216041", "g000476"
]

display(
    df_selected[
        df_selected["name"].isin(reviewer_instances)
        & (df_selected["heuristic"] == "BURER2002")
    ][["name", "limit", "file_limit", "file"]]
    .drop_duplicates()
    .sort_values("name")
)

,name,limit,file_limit,file
2250,g000283,36.585,36.585,./BURER2002/g000283-36.585-5.csv
2685,g000377,7.008,7.008,./BURER2002/g000377-7.008-5.csv
3150,g000476,57.363,57.363,./BURER2002/g000476-57.363-5.csv
5580,g000989,2.874,2.874,./BURER2002/g000989-2.874-5.csv
6055,g001086,8.818,8.818,./BURER2002/g001086-8.818-5.csv
6900,g001269,5.084,5.084,./BURER2002/g001269-5.084-5.csv
7220,g001337,18.048,18.048,./BURER2002/g001337-18.048-5.csv
9780,g001883,24.803,24.803,./BURER2002/g001883-24.803-5.csv
12390,g002440,11.756,11.756,./BURER2002/g002440-11.756-5.csv
13005,g002569,5.388,5.388,./BURER2002/g002569-5.388-5.csv


In [7]:
import numpy as np
import pandas as pd

# Add graph sizes to the cleaned dataset
df_runtime_size = df_selected.merge(
    df_baseline[["name", "nodes"]],
    on="name",
    how="left",
    validate="many_to_one"
) if "nodes" not in df_selected.columns else df_selected.copy()

df_runtime_size["runtime_ratio"] = (
    df_runtime_size["runtime"] / df_runtime_size["limit"]
)

df_runtime_size["size_cat"] = pd.cut(
    df_runtime_size["nodes"],
    bins=[0, 1024, 2048, 4096, 8192, np.inf],
    labels=["xsmall", "small", "medium", "large", "xlarge"],
    right=False
)

size_summary = (
    df_runtime_size.groupby(
        ["heuristic", "size_cat"],
        observed=True
    )
    .agg(
        runs=("runtime", "size"),
        instances=("name", "nunique"),
        median_ratio=("runtime_ratio", "median"),
        p95_ratio=("runtime_ratio", lambda x: x.quantile(0.95)),
        over_10pct=("runtime_ratio", lambda x: (x > 1.10).sum()),
        over_2x=("runtime_ratio", lambda x: (x > 2).sum())
    )
    .reset_index()
)

display(size_summary.round(4))

,heuristic,size_cat,runs,instances,median_ratio,p95_ratio,over_10pct,over_2x
0,BURER2002,xsmall,11735,2347,1.0082,1.0385,30,0
1,BURER2002,small,1160,232,1.0026,1.0151,0,0
2,BURER2002,medium,2045,409,1.0019,1.0205,4,0
3,BURER2002,large,1655,331,1.0018,1.0199,1,0
4,BURER2002,xlarge,385,77,1.0004,1.0022,0,0
5,DAv2,xsmall,11735,2347,0.9984,4.0641,2291,1217
6,DAv2,small,1160,232,0.9941,1.0023,0,0
7,DAv2,medium,2045,409,0.9897,0.9960,10,10
8,DAv2,large,1655,331,0.9894,0.9973,0,0
9,DAv3,xsmall,11735,2347,14.7806,478.2766,10194,9458


In [10]:
df_primary = df_runtime_size[
    (df_runtime_size["nodes"] >= 2048)
    & (df_runtime_size["limit"] > 0.25)
].copy()

primary_summary = (
    df_primary.groupby("heuristic")
    .agg(
        instances=("name", "nunique"),
        runs=("runtime", "size"),
        over_5pct=("runtime_ratio", lambda x: (x > 1.05).sum()),
        over_10pct=("runtime_ratio", lambda x: (x > 1.10).sum()),
        over_2x=("runtime_ratio", lambda x: (x > 2).sum()),
        median_ratio=("runtime_ratio", "median"),
        p95_ratio=("runtime_ratio", lambda x: x.quantile(0.95)),
        max_ratio=("runtime_ratio", "max")
    )
)

display(primary_summary.round(4))

,instances,runs,over_5pct,over_10pct,over_2x,median_ratio,p95_ratio,max_ratio
heuristic,,,,,,,,
BURER2002,815,4075,15,5,0,1.0016,1.0195,1.1462
DAv2,738,3690,0,0,0,0.9896,0.9965,1.0118
DAv3,819,4095,383,38,0,1.0080,1.0651,1.5046
MERZ1999GLS,819,4095,0,0,0,1.0002,1.0012,1.0104
PALUBECKIS2004bMST2,819,4095,0,0,0,1.0031,1.0174,1.0396
